# AIPI 510 – Week 4 Data Exploration
## Welltory COVID‑19 and Wearables – EDA
**Author:** Atharva Jog  
**Created:** 2025-09-16 16:38:01  

This notebook follows the assignment rubric: context & sampling, data understanding, descriptive statistics, missingness & quality, visual exploration, transformations, feature engineering (≥1 new feature), and rationale for every choice.  

> **Note:** Place raw data under `../data/welltory/` (excluded from git). If you cloned the Welltory repo elsewhere, create a symlink into `../data/`.

## 0. Dataset context & sampling (write first)
- **Data source:** Welltory COVID‑19 & Wearables Open Data Research
- **Collection window:** *(fill in from README of the dataset)*
- **Unit of observation:** *(e.g., per‑measurement / per‑user‑per‑day)*
- **Sampling considerations:** *(who is included/excluded, app users, device types, biases)*
- **Data lineage:** *(where I got the files, any transforms I performed before this analysis)*
- **Ethics/privacy:** *(consent summary, anonymization, any sensitive fields & how handled)*

In [ ]:
# 1. Imports & setup
import os, glob, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
pd.set_option('display.max_colwidth', 120)

# Paths
DATA_DIR = os.path.abspath(os.path.join('data', 'welltory'))
DATA_GLOB = os.path.join(DATA_DIR, '**', '*.csv')  # recursive read
DATA_GLOB

In [ ]:
# 2. Load helper – memory‑friendly concatenation of many CSVs (if applicable)
def read_many_csv(paths, n_rows=None):
    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p, low_memory=False, nrows=n_rows)
            df['__source_file'] = os.path.basename(p)
            frames.append(df)
        except Exception as e:
            print('Failed to read', p, e)
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

csv_paths = glob.glob(DATA_GLOB, recursive=True)
len(csv_paths), csv_paths[:5]

In [ ]:
# 3. Load a *primary* table (choose what you want to focus on)
#    Example: pick files that contain 'hrv' or 'covid' in filename.
primary_paths = [p for p in csv_paths if 'hrv' in os.path.basename(p).lower() or 'covid' in os.path.basename(p).lower()]
if not primary_paths:
    primary_paths = csv_paths  # fallback to all

raw = read_many_csv(primary_paths)
raw.shape, raw.head(3)

## 4. Basic profiling & data dictionary draft
Explain what each column likely represents; if unclear, search the dataset docs and record notes *here*. Keep an explicit table of definitions.

In [ ]:
def summarize_df(df: pd.DataFrame):
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().mean() * 100).round(2),
        'n_unique': df.nunique(dropna=True)
    })
    summary['example_values'] = [df[c].dropna().unique()[:5] for c in df.columns]
    return summary.sort_values(['pct_missing','n_unique'], ascending=[False, True])

profile = summarize_df(raw)
profile.head(20)

In [ ]:
# 5. Duplicates & time coverage (if a timestamp column exists)
dup_rows = raw.duplicated().sum()
print('Duplicate rows:', dup_rows)

# Try to parse likely datetime columns
time_cols = [c for c in raw.columns if 'time' in c.lower() or 'date' in c.lower()]
for c in time_cols:
    try:
        raw[c] = pd.to_datetime(raw[c], errors='coerce')
    except Exception:
        pass
raw[time_cols].head(3) if time_cols else 'No columns that look like datetimes discovered'

In [ ]:
# 6. Descriptive statistics by type
num_cols = raw.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in raw.columns if c not in num_cols]
print('Numeric cols (first 10):', num_cols[:10])
print('Categorical/other cols (first 10):', cat_cols[:10])

desc_num = raw[num_cols].describe().T if num_cols else pd.DataFrame()
desc_num.head()

In [ ]:
# 7. Missingness visuals
missing_pct = raw.isna().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct>0]
plt.figure(figsize=(10,6))
missing_pct.head(30).plot(kind='bar')
plt.title('Top columns by missing %')
plt.ylabel('Percent missing')
plt.show()

In [ ]:
# 8. Correlations & outliers (numerics only)
if len(num_cols) >= 2:
    corr = raw[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect='auto')
    plt.colorbar(); plt.title('Correlation (numerics)'); plt.xticks(range(len(num_cols)), num_cols, rotation=90); plt.yticks(range(len(num_cols)), num_cols)
    plt.tight_layout(); plt.show()

    # Simple outlier flag via z-score threshold
    from scipy.stats import zscore
    z = raw[num_cols].apply(zscore, nan_policy='omit')
    outlier_counts = (np.abs(z) > 3).sum().sort_values(ascending=False)
    outlier_counts.head(10) if not outlier_counts.empty else 'No outlier candidates found'
else:
    print('Not enough numeric columns for correlation/outlier analysis.')

## 9. Transformations & rationale
Describe what transformations are justified (e.g., winsorize extreme HRV values, log‑transform skewed variables, parse timestamps to local date, drop impossible values). Provide a **why** for each.

In [ ]:
# Example transforms (edit/replace to match chosen columns)
transformed = raw.copy()

# Example: one‑hot encode a categorical field if present
candidate_cat = None
for c in raw.columns:
    if raw[c].dtype == 'object' and raw[c].nunique(dropna=True) < 20:
        candidate_cat = c; break
if candidate_cat:
    transformed = pd.get_dummies(transformed, columns=[candidate_cat], drop_first=True)
    print('One‑hot encoded:', candidate_cat)

# Example: create at least one new feature – rolling mean by user over time
user_col = next((c for c in raw.columns if 'user' in c.lower() or 'id' in c.lower()), None)
value_col = next((c for c in num_cols if 'hrv' in c.lower() or 'rmssd' in c.lower() or 'bpm' in c.lower()), None)
time_col  = next((c for c in time_cols if pd.api.types.is_datetime64_any_dtype(raw[c])), None)

if all([user_col, value_col, time_col]):
    df_sorted = transformed.sort_values([user_col, time_col])
    transformed['feat_rolling_mean_7'] = df_sorted.groupby(user_col)[value_col].transform(lambda s: s.rolling(7, min_periods=3).mean())
    print('Engineered feature: feat_rolling_mean_7 from', value_col)
else:
    print('Could not find obvious user/value/time columns – replace this block with feature(s) you design.')

transformed.head(3)

In [ ]:
# 10. Save small artifacts for the report (do NOT commit raw data)
ART_DIR  = os.path.abspath(os.path.join('artifacts'))
os.makedirs(ART_DIR, exist_ok=True)
profile.to_csv(os.path.join(ART_DIR, 'profile_summary.csv'), index=True)
desc_num.to_csv(os.path.join(ART_DIR, 'numeric_describe.csv'), index=True)
print('Artifacts saved to', ART_DIR)

## 11. Findings & Next Steps
- Key insights from visuals and stats.
- Data quality issues identified and how they were handled.
- What the new feature adds to the dataset (interpretability, stability, predictive power?).
- What other data *could* we add (future work).